In [15]:
# # ĐỒ ÁN MÔN HỌC: DỰ ĐOÁN TRỄ HẠN GIAO HÀNG (DELIVERY DELAY PREDICTION)

# ## 1. Instruction list: Raw to Tidy
# Quá trình tiền xử lý dữ liệu bao gồm:
# 1. **Gộp dữ liệu:** Tích hợp dữ liệu theo chu kỳ (A: Tháng 4-6, B: Tháng 7-9) và lấy giao tập tính năng (Intersection) để tránh Data Leakage.
# 2. **Missing Data:** Xử lý theo cơ chế MCAR (điền Mode), MAR (tạo Missing Flag), MNAR (Logic Default = 0).
# 3. **Feature Engineering:** Trích xuất Time features từ `Order date`.
# 4. **Feature Encoding:** Áp dụng **Target Encoding** cho các biến High Cardinality (`PRODUCT_CD`, `CUST_CD`, `SUPPLIER_CD`) để tránh bùng nổ chiều dữ liệu.

In [16]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, f1_score

In [ ]:
path = r'D:\\Ds108\\Lab4\\raw_data\\Data for practice\\'

df_delay46 = pd.read_csv(path + 'delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv')
df_notdelay46 = pd.read_csv(path + 'not_delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv')
df_delay79 = pd.read_csv(path + 'delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv')
df_notdelay79 = pd.read_csv(path + 'not_delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv')



common_cols = df_A.columns.intersection(df_B.columns)
df_A = df_A[common_cols].copy()
df_B = df_B[common_cols].copy()

# 2. Thiết kế Pipeline Tiền xử lý (Fit & Transform)

In [18]:
TARGET_ENC_COLS = ['PRODUCT_CD', 'CUST_CD', 'SUPPLIER_CD', 'BRAND_CD']

def preprocess(df, stats=None, is_train=True):
    df = df.copy()

    # ===== DROP LEAKAGE =====
    drop_cols = [
        'SHIP DECISION NO','SOUF_RCV_NO','QTUF_RCV_NO',
        'REASON_CD','ACTUAL_SHIP_DAYS',
        'GLOBAL_NO','INNER_CD','Sales order line number'
    ]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    # ===== TIME =====
    df['Order date'] = pd.to_datetime(df['Order date'], errors='coerce')
    df['month'] = df['Order date'].dt.month
    df['dow'] = df['Order date'].dt.dayofweek
    df = df.drop(columns=['Order date'])

    # ===== MISSING =====
    df['Ship_Mode_missing'] = df['Ship Mode'].isna().astype('int8')
    df['Ship Mode'] = df['Ship Mode'].fillna('Unknown')

    df['OTHER_AREA_FLAG'] = df['OTHER AREA SHIP DIV'].isna().astype('int8')
    df['OTHER AREA SHIP DIV'] = df['OTHER AREA SHIP DIV'].fillna(0)

    # ===== TRAIN: FIT STATS =====
    if is_train:
        stats = {}
        stats['global_mean'] = df['label'].mean()
        stats['mode_SUPPLIER_DIV'] = df['SUPPLIER_DIV'].mode()[0]

        stats['te'] = {}
        for col in TARGET_ENC_COLS:
            if col in df.columns:
                stats['te'][col] = df.groupby(col)['label'].mean()

    # ===== APPLY =====
    df['SUPPLIER_DIV'] = df['SUPPLIER_DIV'].fillna(stats['mode_SUPPLIER_DIV'])

    for col in TARGET_ENC_COLS:
        if col in df.columns:
            df[col] = df[col].map(stats['te'][col])
            df[col] = df[col].fillna(stats['global_mean'])

    return df, stats

In [19]:
def reduce_memory(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        elif df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

# 3 & 4. Xây dựng lõi Thực nghiệm (Partition, Selection & Modeling)
Sử dụng phương pháp **Embedded Feature Selection** (Dùng LightGBM để lấy Top đặc trưng quan trọng) và **Ensemble Learning** (Voting Classifier giữa LightGBM và Logistic Regression / KNN).

In [20]:
def train_cv(df):

    print("🚀 Preprocessing...")
    df, stats = preprocess(df, is_train=True)

    X = df.drop(columns=['label'])
    y = df['label']

    X = reduce_memory(X)

    print("📊 Shape:", X.shape)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n========== FOLD {fold} ==========")

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.03,
            num_leaves=64,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric='auc',
            verbose=200
        )

        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred, average='macro')

        print(f"F1 Fold {fold}: {f1:.4f}")
        scores.append(f1)

    print("\n🔥 CV RESULT")
    print("Mean F1:", np.mean(scores))

    return model, stats

In [23]:
def evaluate(train_df, test_df):

    print("\n🚀 TRAIN → TEST EVALUATION")

    train, stats = preprocess(train_df, is_train=True)
    test, _ = preprocess(test_df, stats=stats, is_train=False)

    X_train = reduce_memory(train.drop(columns=['label']))
    y_train = train['label']

    X_test = reduce_memory(test.drop(columns=['label']))
    y_test = test['label']

    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42
    )

    print("🔥 Training...")
    model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        lgb.log_evaluation(200),     # in log mỗi 200 round
        lgb.early_stopping(100)      # early stopping
    ]
)
    print("📈 Predicting...")
    y_pred = model.predict(X_test)

    print("\n=== RESULT ===")
    print(classification_report(y_test, y_pred))
    print("F1:", f1_score(y_test, y_pred, average='macro'))

In [ ]:
# CV trên toàn bộ dữ liệu
model, stats = train_cv(pd.concat([df_A, df_B]))

# Test A → B
evaluate(df_A, df_B)

# Test B → A
evaluate(df_B, df_A)

In [25]:
# ============================================================
# BASELINE PIPELINE: DỰ ĐOÁN TRỄ HẠN GIAO HÀNG
# Models: LightGBM + KNN (Voting Ensemble)
# Experiments: 4 experiments theo yêu cầu đồ án
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier

# ============================================================
# 0. CẤU HÌNH
# ============================================================
PATH = r'D:\\Ds108\\Lab4\\raw_data\\Data for practice\\'

TARGET_ENC_COLS = ['PRODUCT_CD', 'CUST_CD', 'SUPPLIER_CD', 'BRAND_CD']

LEAKAGE_COLS = [
    'SHIP DECISION NO', 'SOUF_RCV_NO', 'QTUF_RCV_NO',
    'REASON_CD', 'ACTUAL_SHIP_DAYS',
    'GLOBAL_NO', 'INNER_CD', 'Sales order line number'
]

# ============================================================
# 1. ĐỌC & GỘP DỮ LIỆU
# ============================================================
def load_data(path):
    df_delay46    = pd.read_csv(path + 'delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv')
    df_notdelay46 = pd.read_csv(path + 'not_delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv')
    df_delay79    = pd.read_csv(path + 'delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv')
    df_notdelay79 = pd.read_csv(path + 'not_delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv')

    df_A = pd.concat([df_delay46, df_notdelay46], ignore_index=True)
    df_B = pd.concat([df_delay79, df_notdelay79], ignore_index=True)

    # Lấy giao tập cột để tránh schema mismatch
    common_cols = df_A.columns.intersection(df_B.columns)
    df_A = df_A[common_cols].copy()
    df_B = df_B[common_cols].copy()

    print(f"✅ df_A (Tháng 4-6): {df_A.shape} | Label dist: {df_A['label'].value_counts().to_dict()}")
    print(f"✅ df_B (Tháng 7-9): {df_B.shape} | Label dist: {df_B['label'].value_counts().to_dict()}")
    return df_A, df_B


# ============================================================
# 2. PREPROCESSING (Fit + Transform tách biệt, tránh leakage)
# ============================================================
def fit_preprocess(df_train):
    """Học tham số từ tập Train"""
    stats = {}
    stats['global_mean']       = df_train['label'].mean()
    stats['mode_SUPPLIER_DIV'] = df_train['SUPPLIER_DIV'].mode()[0] if 'SUPPLIER_DIV' in df_train.columns else 0

    # Target Encoding: tỉ lệ delay theo từng mã
    stats['te'] = {}
    for col in TARGET_ENC_COLS:
        if col in df_train.columns:
            stats['te'][col] = df_train.groupby(col)['label'].mean().to_dict()

    # Lưu lại tên cột sau get_dummies để align Test
    stats['train_cols'] = None  # sẽ set sau transform
    return stats


def transform_preprocess(df, stats):
    """Áp dụng tham số đã học vào bất kỳ tập nào"""
    df = df.copy()

    # --- DROP LEAKAGE & ID ---
    df = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns], errors='ignore')

    # --- TIME FEATURES ---
    if 'Order date' in df.columns:
        df['Order date'] = pd.to_datetime(df['Order date'], errors='coerce')
        df['Order_Month']     = df['Order date'].dt.month
        df['Order_Day']       = df['Order date'].dt.day
        df['Order_DayOfWeek'] = df['Order date'].dt.dayofweek
        df = df.drop(columns=['Order date'])

    # --- MISSING DATA ---
    if 'Ship Mode' in df.columns:
        df['Ship_Mode_missing'] = df['Ship Mode'].isna().astype('int8')
        df['Ship Mode']         = df['Ship Mode'].fillna('Unknown')

    if 'OTHER AREA SHIP DIV' in df.columns:
        df['OTHER_AREA_FLAG']       = df['OTHER AREA SHIP DIV'].isna().astype('int8')
        df['OTHER AREA SHIP DIV']   = df['OTHER AREA SHIP DIV'].fillna(0)

    if 'SUPPLIER_DIV' in df.columns:
        df['SUPPLIER_DIV'] = df['SUPPLIER_DIV'].fillna(stats['mode_SUPPLIER_DIV'])

    if 'WEIGHT_UNIT' in df.columns:
        df['WEIGHT_UNIT'] = df['WEIGHT_UNIT'].fillna('Unknown')

    # --- TARGET ENCODING (High Cardinality) ---
    for col in TARGET_ENC_COLS:
        if col in df.columns:
            df[col] = df[col].map(stats['te'].get(col, {})).fillna(stats['global_mean'])

    # --- ONE-HOT ENCODING (Low Cardinality còn lại) ---
    df = pd.get_dummies(df)

    # --- XÓA CỘT TRÙNG (tránh lỗi LightGBM) ---
    df = df.loc[:, ~df.columns.duplicated()]

    return df


def reduce_memory(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        elif df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df


# ============================================================
# 3. XÂY DỰNG MODEL (LightGBM + KNN Voting Ensemble)
# ============================================================
def build_model():
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    knn_pipeline = Pipeline([
        ('scaler', StandardScaler()),          # KNN cần scale
        ('knn', KNeighborsClassifier(
            n_neighbors=11,
            weights='distance',
            metric='euclidean',
            n_jobs=-1
        ))
    ])

    ensemble = VotingClassifier(
        estimators=[
            ('LightGBM', lgb_clf),
            ('KNN',      knn_pipeline)
        ],
        voting='soft',      # dùng xác suất để vote
        weights=[2, 1]      # LightGBM đáng tin hơn, weight cao hơn
    )

    return ensemble


# ============================================================
# 4. PIPELINE TRAIN + EVALUATE
# ============================================================
def run_experiment(train_df, test_df, exp_name=""):
    """
    Nhận raw train/test DataFrame,
    trả về macro F1 trên tập test.
    """
    print(f"\n{'='*55}")
    print(f"  {exp_name}")
    print(f"{'='*55}")

    # Fit stats từ train
    stats = fit_preprocess(train_df)

    # Transform cả hai tập
    train_clean = transform_preprocess(train_df, stats)
    test_clean  = transform_preprocess(test_df,  stats)

    # Tách X, y
    X_train = reduce_memory(train_clean.drop(columns=['label']))
    y_train = train_clean['label']
    X_test  = reduce_memory(test_clean.drop(columns=['label']))
    y_test  = test_clean['label']

    # Align cột (test có thể thiếu 1 số cột dummies)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    # Build & Train
    print(f"  Train: {X_train.shape} | Test: {X_test.shape}")
    model = build_model()
    model.fit(X_train, y_train)

    # Predict & Score
    y_pred   = model.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    acc      = accuracy_score(y_test, y_pred)

    print(classification_report(y_test, y_pred, target_names=['Not Delay', 'Delay']))
    print(f"  >> Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}")

    return macro_f1


# ============================================================
# 5. 4 EXPERIMENTS
# ============================================================
def main():
    # --- Load ---
    df_A, df_B = load_data(PATH)
    results = {}

    # ----------------------------------------------------------
    # EXP 1: Train A → Test B
    # ----------------------------------------------------------
    results['Exp1 (A→B)'] = run_experiment(
        train_df=df_A, test_df=df_B,
        exp_name="EXP 1: Train (Tháng 4-6) → Test (Tháng 7-9)"
    )

    # ----------------------------------------------------------
    # EXP 2: Train B → Test A
    # ----------------------------------------------------------
    results['Exp2 (B→A)'] = run_experiment(
        train_df=df_B, test_df=df_A,
        exp_name="EXP 2: Train (Tháng 7-9) → Test (Tháng 4-6)"
    )

    # ----------------------------------------------------------
    # EXP 3: 5-Fold CV trên toàn bộ A+B
    # ----------------------------------------------------------
    print(f"\n{'='*55}")
    print(f"  EXP 3: 5-Fold Stratified CV (A + B)")
    print(f"{'='*55}")

    df_AB = pd.concat([df_A, df_B], ignore_index=True)
    skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(df_AB, df_AB['label']), 1):
        f1_fold = run_experiment(
            train_df=df_AB.iloc[train_idx],
            test_df =df_AB.iloc[val_idx],
            exp_name=f"  Fold {fold}/5"
        )
        fold_scores.append(f1_fold)

    results['Exp3 (5-Fold CV)'] = np.mean(fold_scores)
    print(f"\n  >> MEAN Macro F1 (5 Folds): {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

    # ----------------------------------------------------------
    # EXP 4: Train (A + k%B) → Test ((1-k)%B), k ∈ {0.1,...,0.9}
    # ----------------------------------------------------------
    print(f"\n{'='*55}")
    print(f"  EXP 4: Increasing k (Train A + k%B → Test (1-k)%B)")
    print(f"{'='*55}")

    k_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]

    for k in k_ratios:
        df_B_train, df_B_test = train_test_split(
            df_B, train_size=k, stratify=df_B['label'], random_state=42
        )
        df_train_k = pd.concat([df_A, df_B_train], ignore_index=True)

        f1_k = run_experiment(
            train_df=df_train_k,
            test_df =df_B_test,
            exp_name=f"EXP 4 | k={int(k*100)}%: Train (A + {int(k*100)}%B) → Test ({int((1-k)*100)}%B)"
        )
        results[f'Exp4 (k={int(k*100)}%)'] = f1_k

    # ----------------------------------------------------------
    # TỔNG HỢP KẾT QUẢ
    # ----------------------------------------------------------
    print(f"\n\n{'='*55}")
    print("  TỔNG HỢP MACRO F1-SCORE")
    print(f"{'='*55}")
    for exp, score in results.items():
        print(f"  {exp:<28}: {score:.4f}")

    return results


# ============================================================
# ENTRY POINT
# ============================================================
if __name__ == '__main__':
    results = main()

MemoryError: Unable to allocate 77.2 MiB for an array with shape (26, 389320) and data type int64